# Prepare the data

In [82]:
# sotify app uri: http://127.0.0.1:9090

import spotipy
import os
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv

load_dotenv()

client_id = os.getenv("SPOTIFY_CLIENT_ID")
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=client_id,
                                               client_secret=client_secret,
                                               redirect_uri="http://127.0.0.1:9090",
                                               scope="user-library-read"))



In [83]:
# Load the data
import pandas as pd
import numpy as np

data = pd.read_csv('../data/raw/dataset.csv', na_values=["", "NaN", "-", "null", "Null", "none", "None"])


# Drop NaNs and irrelevant columns, define features array and target variable
data = data.dropna()
data.drop_duplicates(inplace=True)

In [84]:
# Filter out all the samples that are copies of the original song
artist_med = data.groupby('artists')['popularity'].transform('median')
junk = (artist_med > 30) & (data['popularity'] < 10)

clean_data = data[~junk].copy()

clean_data.info()
# TODO: the last impute line leak a tiny bit of information - for the train/test split, since it uses the whole dataset for average

<class 'pandas.DataFrame'>
Index: 112338 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          112338 non-null  int64  
 1   spotify_track_id  112338 non-null  str    
 2   artists           112338 non-null  str    
 3   album_name        112338 non-null  str    
 4   track_name        112338 non-null  str    
 5   popularity        112338 non-null  int64  
 6   duration_ms       112338 non-null  int64  
 7   explicit          112338 non-null  bool   
 8   danceability      112338 non-null  float64
 9   energy            112338 non-null  float64
 10  key               112338 non-null  int64  
 11  loudness          112338 non-null  float64
 12  mode              112338 non-null  int64  
 13  speechiness       112338 non-null  float64
 14  acousticness      112338 non-null  float64
 15  instrumentalness  112338 non-null  float64
 16  liveness          112338 non-null  f

In [85]:
# explode to one row per (track, individual artist); original index preserved
df = clean_data[['artists', 'popularity']].copy()
df['artist'] = df['artists'].str.split(';')
ex = df.explode('artist')
ex['artist'] = ex['artist'].str.strip()

# per-row leave-one-out median within each artist
def loo_median(s):
    vals = s.to_numpy()
    n = vals.size
    if n == 1:                       # single track -> no "other" tracks
        return np.array([np.nan])
    return np.array([np.median(np.delete(vals, i)) for i in range(n)])

ex['artist_loo_med'] = ex.groupby('artist')['popularity'].transform(loo_median)

# collapse back to track level: max across the track's artists (skips NaN)
fame = ex.groupby(level=0)['artist_loo_med'].max()
clean_data['artist_fame_loo'] = fame

# tracks where every artist had only one song -> no signal -> neutral fill
clean_data['artist_fame_loo'] = clean_data['artist_fame_loo'].fillna(
    clean_data['popularity'].median()
)

data = clean_data.copy()

In [88]:
clean_data = clean_data.drop(columns=["track_id", "spotify_track_id", "artists", "album_name", "track_name"])

In [ ]:
clean_data.to_parquet('../data/interim/tracks_enriched.parquet')
data.to_parquet('../data/interim/orig_data.parquet')